# 01 Build Gold Forecasting Base Table

This notebook constructs `gold_forecasting_base.parquet` — a leakage-free, one-row-per-`(forecast_date, store_id, sku_id)` table where the target is the **actual unit sales on the next calendar day**.

## Design Decisions

### Technology: DuckDB + PyArrow
The `sales_fact` table has **125.5M rows**. Loading it into Pandas twice (once for today's features, once for the target lookup) would require 16–24 GB of RAM. Instead, DuckDB queries the Parquet files directly on disk and streams the join — the full 125M row self-join never materialises in Python memory. PyArrow handles file I/O.

### Target Construction: Explicit Self-Join (not `.shift(-1)`)
`.shift(-1)` silently bridges gaps: if a SKU has no record on Aug 16, it would assign Aug 17's sales as Aug 15's target. Our approach: inner-join on `target_date = forecast_date + 1 day`. If the exact next calendar date does not exist for that `(store_id, sku_id)`, the row is dropped — not silently filled.

### Modeling Assumption: Promotions as Known-Future Features
`on_promotion_tomorrow` is included as a feature. **Assumption:** promotional schedules are planned and communicated to store managers in advance. At the time the order decision is made (today), tomorrow's promotion status is a legitimately knowable feature — not data leakage. This is standard in retail forecasting.

### Oil Price: Joined on `forecast_date`
Today's oil price is known at decision time. Tomorrow's is not. Therefore `oil_price` is joined on `forecast_date`.

## Pipeline Position
```
Silver (5 tables)
  ↓
Gold Forecasting Base  ← THIS NOTEBOOK
  ↓
Gold Feature Engineering (lag/rolling)
  ↓
LightGBM → Newsvendor → Order Quantity
```

In [1]:
%pip install duckdb pyarrow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
from pathlib import Path

# ── Dynamic lakehouse root resolution (same pattern as Silver notebooks) ──────
lakehouse_root = None
for p in [Path.cwd()] + list(Path.cwd().parents):
    if (p / "data_lake").is_dir() and (p / "notebooks").is_dir():
        lakehouse_root = p
        break

if lakehouse_root is None:
    lakehouse_root = Path("D:/Demand_forecasting/inventory-lakehouse")

SILVER_DIR = lakehouse_root / "data_lake" / "silver"
GOLD_DIR   = lakehouse_root / "data_lake" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Lakehouse root : {lakehouse_root}")
print(f"Silver dir     : {SILVER_DIR}")
print(f"Gold dir       : {GOLD_DIR}")
print(f"DuckDB version : {duckdb.__version__}")
print("Libraries loaded successfully!")

Lakehouse root : d:\Demand_forecasting\inventory-lakehouse
Silver dir     : d:\Demand_forecasting\inventory-lakehouse\data_lake\silver
Gold dir       : d:\Demand_forecasting\inventory-lakehouse\data_lake\gold
DuckDB version : 1.5.5
Libraries loaded successfully!


## Step 1 — Validate Silver Paths
Confirm all 5 Silver tables exist before attempting any joins.

In [3]:
# ── Resolve all Silver paths ──────────────────────────────────────────────────
SALES_GLOB  = str(SILVER_DIR / "sales_fact"          / "*.parquet")
ITEMS_PATH  = str(SILVER_DIR / "item_dimension"      / "item_dimension.parquet")
STORES_PATH = str(SILVER_DIR / "store_dimension"     / "store_dimension.parquet")
CAL_PATH    = str(SILVER_DIR / "calendar_dimension"  / "calendar_dimension.parquet")
OIL_PATH    = str(SILVER_DIR / "oil_dimension"       / "oil_dimension.parquet")

# Verify all expected files/directories exist
checks = {
    "sales_fact (dir)"    : (SILVER_DIR / "sales_fact").is_dir(),
    "item_dimension"      : Path(ITEMS_PATH).is_file(),
    "store_dimension"     : Path(STORES_PATH).is_file(),
    "calendar_dimension"  : Path(CAL_PATH).is_file(),
    "oil_dimension"       : Path(OIL_PATH).is_file(),
}

all_ok = True
for name, exists in checks.items():
    status = "\u2705" if exists else "\u274c MISSING"
    print(f"  {status}  {name}")
    if not exists:
        all_ok = False

assert all_ok, "One or more Silver tables are missing. Run Silver notebooks first."
print("\n\u2705 All Silver tables found. Proceeding.")

  ✅  sales_fact (dir)
  ✅  item_dimension
  ✅  store_dimension
  ✅  calendar_dimension
  ✅  oil_dimension

✅ All Silver tables found. Proceeding.


## Step 2 — Uniqueness Validation

Before the self-join we must verify that `(sales_date, store_id, sku_id)` is a **unique key** in `sales_fact`.

A duplicate key would silently multiply rows in the output — the self-join would produce more rows than expected with no error. We detect this **before** the join using DuckDB (no RAM cost — queries Parquet directly on disk).

In [4]:
import time

con = duckdb.connect()

print("Checking (sales_date, store_id, sku_id) uniqueness in sales_fact...")
print("(This scans 125.5M rows via DuckDB — may take 30-60 seconds.)")

t0 = time.time()
dup_count = con.execute(f"""
    SELECT COUNT(*) AS duplicate_count
    FROM (
        SELECT sales_date, store_id, sku_id, COUNT(*) AS cnt
        FROM read_parquet('{SALES_GLOB}')
        GROUP BY sales_date, store_id, sku_id
        HAVING cnt > 1
    )
""").fetchone()[0]
elapsed = time.time() - t0

print(f"Duplicate (sales_date, store_id, sku_id) keys: {dup_count:,}  [{elapsed:.1f}s]")

assert dup_count == 0, (
    f"STOP: {dup_count:,} duplicate composite keys found in sales_fact. "
    "Investigate before joining."
)

print("\u2705 Uniqueness validated. Safe to proceed with self-join.")

Checking (sales_date, store_id, sku_id) uniqueness in sales_fact...
(This scans 125.5M rows via DuckDB — may take 30-60 seconds.)
Duplicate (sales_date, store_id, sku_id) keys: 0  [3.6s]
✅ Uniqueness validated. Safe to proceed with self-join.


## Step 3 — Build Gold Forecasting Base via DuckDB SQL

All joins happen in a single SQL query. DuckDB reads Parquet files directly — the 125M row fact table is never loaded into Python RAM.

**Join logic:**
```
today (sales_fact)             -> forecast_date features (known now)
   INNER JOIN tomorrow (self)  -> on target_date = forecast_date + 1 day  -> target_unit_sales
   LEFT JOIN  item_dimension   -> on sku_id       -> family_en AS family, is_perishable
   LEFT JOIN  store_dimension  -> on store_id     -> city, store_type, cluster
   LEFT JOIN  calendar         -> on target_date  -> tomorrow's calendar context
   LEFT JOIN  oil_dimension    -> on forecast_date -> oil_price (known today)
```

> **Note on `family` column:** `item_dimension` stores the product family in two columns: `family_en` (English) and `family_es` (Spanish). We use `family_en` aliased as `family`.

In [5]:
# Cell 3 — Build Gold Forecasting Base via DuckDB SQL
# CHANGES vs previous version:
#   1. COALESCE(is_promotion, false) in both CTEs — treats NaN promotions as False
#   2. .to_arrow_table() instead of .fetch_arrow_table() — fixes deprecation warning

print("Building Gold forecasting base via DuckDB...")
print("(Full self-join over 125.5M rows — may take 2-5 minutes.)")

t0 = time.time()

sql = f"""
    WITH today AS (
        SELECT
            CAST(sales_date AS DATE)             AS forecast_date,
            store_id,
            sku_id,
            units_sold,
            COALESCE(is_promotion, false)        AS on_promotion_today
        FROM read_parquet('{SALES_GLOB}')
    ),

    tomorrow AS (
        SELECT
            CAST(sales_date AS DATE)             AS target_date,
            store_id,
            sku_id,
            units_sold                           AS target_unit_sales,
            COALESCE(is_promotion, false)        AS on_promotion_tomorrow
        FROM read_parquet('{SALES_GLOB}')
    ),

    base AS (
        SELECT
            t.forecast_date,
            (t.forecast_date + INTERVAL '1' DAY)::DATE  AS target_date,
            t.store_id,
            t.sku_id,
            t.units_sold,
            t.on_promotion_today,
            tm.target_unit_sales,
            tm.on_promotion_tomorrow
        FROM today t
        INNER JOIN tomorrow tm
            ON  tm.target_date = t.forecast_date + INTERVAL '1' DAY
            AND tm.store_id    = t.store_id
            AND tm.sku_id      = t.sku_id
    )

    SELECT
        b.forecast_date,
        b.target_date,
        b.store_id,
        b.sku_id,
        b.units_sold,
        b.target_unit_sales,
        i.family_en      AS family,
        i.is_perishable,
        s.city,
        s.store_type,
        s.cluster,
        c.day_of_week    AS target_day_of_week,
        c.month          AS target_month,
        c.is_weekend     AS target_is_weekend,
        c.is_holiday     AS target_is_holiday,
        b.on_promotion_today,
        b.on_promotion_tomorrow,
        o.oil_price

    FROM base b
    LEFT JOIN read_parquet('{ITEMS_PATH}')  i  ON i.sku_id   = b.sku_id
    LEFT JOIN read_parquet('{STORES_PATH}') s  ON s.store_id = b.store_id
    LEFT JOIN read_parquet('{CAL_PATH}')    c  ON c.date     = b.target_date
    LEFT JOIN read_parquet('{OIL_PATH}')    o  ON o.date     = b.forecast_date

    ORDER BY b.forecast_date, b.store_id, b.sku_id
"""

result_arrow = con.execute(sql).to_arrow_table()   # .to_arrow_table() replaces deprecated .fetch_arrow_table()
elapsed = time.time() - t0

print(f"\n✅ Gold base built in {elapsed:.1f}s")
print(f"   Rows    : {result_arrow.num_rows:,}")
print(f"   Columns : {result_arrow.num_columns}")


Building Gold forecasting base via DuckDB...
(Full self-join over 125.5M rows — may take 2-5 minutes.)

✅ Gold base built in 136.8s
   Rows    : 105,951,916
   Columns : 18


## Step 4 — Export to Parquet

In [6]:
out_path = GOLD_DIR / "gold_forecasting_base.parquet"

print(f"Writing to {out_path} ...")
t0 = time.time()

pq.write_table(result_arrow, out_path, compression="snappy")

elapsed = time.time() - t0
file_mb = out_path.stat().st_size / (1024 * 1024)

print(f"\u2705 Saved in {elapsed:.1f}s")
print(f"   Path      : {out_path}")
print(f"   File size : {file_mb:.1f} MB")

Writing to d:\Demand_forecasting\inventory-lakehouse\data_lake\gold\gold_forecasting_base.parquet ...
✅ Saved in 46.7s
   Path      : d:\Demand_forecasting\inventory-lakehouse\data_lake\gold\gold_forecasting_base.parquet
   File size : 545.8 MB


## Step 5 — Validate & Display Sample

Load the full table into Pandas for three validation checks:
1. **Leakage check** — `target_date` is always exactly `forecast_date + 1`.
2. **Schema check** — all 18 expected columns are present.
3. **Null check** — no nulls in critical key/target columns.

In [7]:
# ── Memory-safe validation — no full .to_pandas() on 106M rows ────────────────
# All checks run on the in-memory Arrow table (result_arrow) or via DuckDB SQL.
import pyarrow.compute as pc

tbl = result_arrow   # already in memory from Step 3 (Arrow, not pandas)

# ── 1. Leakage check (Arrow compute — zero copy) ──────────────────────────────
fd = pc.cast(tbl['forecast_date'], pa.date32())
td = pc.cast(tbl['target_date'],   pa.date32())

# duration[s] → int64 (seconds) → divide by 86400 → days
diff_seconds = pc.cast(pc.subtract(td, fd), pa.int64())
diff_days    = pc.divide(diff_seconds, 86400)
gaps_list    = sorted(pc.unique(diff_days).to_pylist())

assert gaps_list == [1], f"❌ Gap check FAILED - found gaps: {gaps_list}"
print("✅ Leakage check  : target_date = forecast_date + 1 for ALL rows.")


# ── 2. Schema check (Arrow schema — no data load) ─────────────────────────────
EXPECTED_COLUMNS = [
    "forecast_date", "target_date",
    "store_id", "sku_id",
    "units_sold", "target_unit_sales",
    "family", "is_perishable",
    "city", "store_type", "cluster",
    "target_day_of_week", "target_month", "target_is_weekend", "target_is_holiday",
    "on_promotion_today", "on_promotion_tomorrow",
    "oil_price",
]
actual_cols = tbl.schema.names
missing_cols = [c for c in EXPECTED_COLUMNS if c not in actual_cols]
assert not missing_cols, f"❌ Missing columns: {missing_cols}"
print(f"✅ Schema check   : all {len(EXPECTED_COLUMNS)} expected columns present.")

# ── 3. Null check via DuckDB (reads saved Parquet — never loads to pandas) ─────
critical = ["forecast_date", "target_date", "store_id", "sku_id",
            "units_sold", "target_unit_sales", "family", "city"]
out_path_fwd = str(out_path).replace("\\", "/")
null_exprs = ", ".join(
    f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}" for c in critical
)
null_row = con.execute(f"SELECT {null_exprs} FROM read_parquet('{out_path_fwd}')").fetchdf()
total_nulls = null_row.values.sum()
if total_nulls == 0:
    print("✅ Null check     : no nulls in critical columns.")
else:
    print("⚠️  Nulls found in critical columns:")
    print(null_row.T[null_row.T[0] > 0])

# ── Summary via DuckDB (aggregates on disk — tiny result set) ──────────────────
summary = con.execute(f"""
    SELECT
        COUNT(*)                 AS total_rows,
        MIN(forecast_date)       AS date_min,
        MAX(forecast_date)       AS date_max,
        COUNT(DISTINCT store_id) AS unique_stores,
        COUNT(DISTINCT sku_id)   AS unique_skus
    FROM read_parquet('{out_path_fwd}')
""").fetchone()

print(f"\n{'─'*50}")
print(f"   Total rows    : {summary[0]:,}")
print(f"   Date range    : {summary[1]} -> {summary[2]}")
print(f"   Unique stores : {summary[3]}")
print(f"   Unique SKUs   : {summary[4]}")
print(f"\n{'─'*50}")
print("Column Dtypes (from Arrow schema):")
for field in tbl.schema:
    print(f"  {field.name:<30} {field.type}")


✅ Leakage check  : target_date = forecast_date + 1 for ALL rows.
✅ Schema check   : all 18 expected columns present.
✅ Null check     : no nulls in critical columns.

──────────────────────────────────────────────────
   Total rows    : 105,951,916
   Date range    : 2013-01-01 -> 2017-08-14
   Unique stores : 54
   Unique SKUs   : 4028

──────────────────────────────────────────────────
Column Dtypes (from Arrow schema):
  forecast_date                  date32[day]
  target_date                    date32[day]
  store_id                       int32
  sku_id                         int32
  units_sold                     double
  target_unit_sales              double
  family                         string
  is_perishable                  bool
  city                           string
  store_type                     string
  cluster                        int32
  target_day_of_week             int32
  target_month                   int32
  target_is_weekend              bool
  target_is_h